In [1]:
import nltk

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# from textblob import TextBlob
from pathlib import Path
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import contractions
from nltk import pos_tag
from nltk.corpus import stopwords
import matplotlib.ticker as ticker
import re
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch
from nltk.corpus import stopwords  
from gensim.models import Word2Vec

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModel, AutoTokenizer
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report

c:\Users\alvar\OneDrive\Escritorio\clean_nlp_env\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def select_text(entry_1, entry_2, entry_3):
    text = entry_2 if entry_3 == 1 else entry_1
    return text

def get_text(df, entry_1, entry_2, entry_3):

    df['final_text'] = df.progress_apply(
        lambda row: select_text(row[entry_1], row[entry_2], row[entry_3]),
        axis=1
    )

    return df


In [3]:
def extract_keyword_info(text, keyword):
    """
    Extracts climate-related keyword information from a text string.

    Parameters:
        text (str): The speech or text in which to search for keywords.
        keywords (list of str): A list of climate-related keywords or phrases to match against the text.
    """
    matches = []
    pattern = r'\b' + re.escape(keyword) + r's?\b' # Match also plural forms
    if re.search(pattern, text, flags=re.IGNORECASE):
        matches.append(keyword)
    contains_keyword = len(matches) > 0
    return pd.Series([matches, contains_keyword])

In [4]:
bigrams = ["climate change", "global warming", "global warm","cap and trade", "paris accord", "emissions trading", "global average temperature", 
            "kyoto protocol", "changing climate" ,"climate resilience","climate decay", "carbon dioxide", "carbon-dioxide","climate politics", 
            "framework on convention climate change", "bali roadmap", "bali action plan", 
            "greenhouse gas", "greenhouse-gas","greenhouse effect", "climate mitigation", "climate action", "emissions", "temperature", "extreme weather", 
            "global environmental change", "global environment", "global environmental" ,"climate variability", "low carbon", "renewable energy", 
            "carbon emission", "climate pollutant", "climate pollutants", "carbon tax", "carbon footprint", "carbon neutrality", 
            "net zero","net-zero","climate crisis", "climate summit", "climate catastrophe", "climate justice", "climate emergency", "climate funding",
            "climate fund", "climate financing", "climate finance","climate peace", "climate agreement", "climate security", "climate ambition", 
            "climate issue", "climate impact", "climate conference", "climate event", "climate challenge", "climate trust", "climate negotiation", 
            "climate catastrophe", "climate risk", "climate goal", "climate change-related", "climate regime", "climate resilient", "climate policy", 
            "carbon market", "carbon sink", "green climate", "green economy", "emission reduction", "emissions reduction", "carbon neutral", 
            "ozone layer", "emissions trading scheme", "co2 emission", "united nation"]

bigrams_1 = sorted(bigrams, key=lambda x: -len(x.split()))

def replace_bigrams(text, bigrams):
    for phrase in bigrams:
        # Normalize the phrase: convert dashes to spaces
        normalized_phrase = phrase.replace("-", " ")
        # Build regex pattern from normalized phrase
        pattern = r'\b' + re.escape(normalized_phrase) +  r's?\b'
        # Replace spaces (now consistent) with underscores
        underscored = normalized_phrase.replace(" ", "_")
        text = re.sub(pattern, underscored, text)
    return text

In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [6]:
def extract_climate_sentences_with_context(speech, keywords):
    """
    Extracts sentences containing climate keywords along with one sentence before and after each.
    Returns a list of context windows (strings of 1–3 sentences).
    """
    doc = list(nlp(speech).sents)  # Convert to list to access by index
    context_windows = []

    for i, sent in enumerate(doc):
        sent_text = sent.text.strip()

        if any(kw.lower() in sent_text.lower() for kw in keywords):
            # Grab the previous, current, and next sentence if available
            prev_sent = doc[i-1].text.strip() if i > 0 else ''
            next_sent = doc[i+1].text.strip() if i < len(doc) - 1 else ''
            
            context = ' '.join([s for s in [prev_sent, sent_text, next_sent] if s])
            context_windows.append(context)

    return context_windows

In [7]:
""" def evaluate_model(model, data_loader, device, return_predictions=False):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            # Forward pass
            logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            
            # Calculate loss
            loss = nn.CrossEntropyLoss()(logits, batch['labels'].to(device))
            total_loss += loss.item()
            
            # Store predictions
            preds = logits.argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = (np.array(all_preds) == np.array(all_labels)).mean()
    
    if return_predictions:
        return avg_loss, accuracy, all_preds, all_labels
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, optimizer, scheduler, device, patience=15, epochs=100):

    class_names = ["Low", "Lower-Middle", "Upper-Middle", "High"]

    best_val_loss = float('inf')
    epochs_no_improve = 0
    training_stats = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(epochs):
        # Training Phase
        model.train()
        epoch_train_loss, epoch_train_acc = 0, 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            
            # Calculate loss
            loss = nn.CrossEntropyLoss()(logits, batch['labels'].to(device))
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            # Update metrics
            epoch_train_loss += loss.item()
            epoch_train_acc += (logits.argmax(-1) == batch['labels'].to(device)).sum().item()

        # Validation Phase
        val_loss, val_acc, _, _ = evaluate_model(model, val_loader, device)
        
        # Store metrics
        training_stats['train_loss'].append(epoch_train_loss / len(train_loader))
        training_stats['train_acc'].append(epoch_train_acc / len(train_loader.dataset))
        training_stats['val_loss'].append(val_loss)
        training_stats['val_acc'].append(val_acc)

        # Checkpointing
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return 'best_model.pth', training_stats """

' def evaluate_model(model, data_loader, device, return_predictions=False):\n    model.eval()\n    total_loss = 0\n    all_preds = []\n    all_labels = []\n\n    with torch.no_grad():\n        for batch in data_loader:\n            # Forward pass\n            logits = model(\n                input_ids=batch[\'input_ids\'].to(device),\n                attention_mask=batch[\'attention_mask\'].to(device)\n            )\n\n            # Calculate loss\n            loss = nn.CrossEntropyLoss()(logits, batch[\'labels\'].to(device))\n            total_loss += loss.item()\n\n            # Store predictions\n            preds = logits.argmax(-1).cpu().numpy()\n            all_preds.extend(preds)\n            all_labels.extend(batch[\'labels\'].cpu().numpy())\n\n    avg_loss = total_loss / len(data_loader)\n    accuracy = (np.array(all_preds) == np.array(all_labels)).mean()\n\n    if return_predictions:\n        return avg_loss, accuracy, all_preds, all_labels\n    return avg_loss, accuracy\n\nd

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, data_loader, criterion, device, num_classes=4):
    model.eval()
    metrics = {'loss': 0, 'correct': 0, 'total': 0, 'all_preds': [], 'all_labels': []}

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            metrics['loss'] += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            metrics['correct'] += (preds == labels).sum().item()
            metrics['total'] += labels.size(0)

            metrics['all_preds'].extend(preds.cpu().numpy())
            metrics['all_labels'].extend(labels.cpu().numpy())

    metrics['loss'] /= metrics['total']
    metrics['acc'] = metrics['correct'] / metrics['total']
    report = classification_report(metrics['all_labels'], metrics['all_preds'], output_dict=True, zero_division=0)
    metrics['f1'] = report['macro avg']['f1-score']
    metrics['conf_matrix'] = confusion_matrix(metrics['all_labels'], metrics['all_preds'], labels=list(range(num_classes)))

    return metrics



def train_model(model, train_loader, val_loader, optimizer, scheduler, device,
                patience=15, epochs=100, num_classes=4):
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')
    epochs_no_improve = 0

    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': [],
        'confusion_matrices': []
    }

    for epoch in range(epochs):
        print("This is the epoch number", epoch)
        model.train()
        train_metrics = {'loss': 0, 'correct': 0, 'total': 0, 'all_preds': [], 'all_labels': []}

        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            train_metrics['loss'] += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            train_metrics['correct'] += (preds == labels).sum().item()
            train_metrics['total'] += labels.size(0)
            train_metrics['all_preds'].extend(preds.cpu().numpy())
            train_metrics['all_labels'].extend(labels.cpu().numpy())

        train_loss = train_metrics['loss'] / train_metrics['total']
        train_acc = train_metrics['correct'] / train_metrics['total']
        train_f1 = classification_report(train_metrics['all_labels'], train_metrics['all_preds'],
                                         output_dict=True, zero_division=0)['macro avg']['f1-score']

        val_metrics = evaluate_model(model, val_loader, criterion, device, num_classes)

        # Save to history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['acc'])
        history['val_f1'].append(val_metrics['f1'])
        history['confusion_matrices'].append(val_metrics['conf_matrix'])

        print(f"Epoch {epoch+1}:")
        print(f"Train  - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
        print(f"Val    - Loss: {val_metrics['loss']:.4f}, Acc: {val_metrics['acc']:.4f}, F1: {val_metrics['f1']:.4f}")
        print("-" * 50)

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            torch.save(model.state_dict(), 'best_model.pth')
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load('best_model.pth'))
    return model, history



In [9]:
def plot_training_history(history, method = None, lr = None):
    plt.figure(figsize=(12, 5))
    
    # Loss plot
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Bert") / f"loss_word2vec_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    plt.legend()
    
    # Accuracy plot
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Bert") / f"acc_word2vec_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    plt.tight_layout()
    #plt.show()
    
    # F1-score plot
    plt.figure(figsize=(6, 4))
    plt.plot(history['train_f1'], label='Train F1')
    plt.plot(history['val_f1'], label='Validation F1')
    plt.title('Training and Validation F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Bert")  / f"f1_word2vec_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    #plt.show()

def plot_confusion_matrix(cm, classes, method = None, lr = None):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plot_path = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Bert") / f"final_model_confusion_matrix_{method}_&_{lr}.png"
    plt.savefig(plot_path)
    #plt.show()

In [10]:
def run_full_experiment(model, train_loader, val_loader, test_loader, device,
                       optimizer = None, scheduler = None, method_name="ClimateBERT", 
                       learning_rate=None, num_classes=4):
    """
    Complete training and evaluation workflow with plotting
    Args:
        method_name: Name of the method (for plot filenames)
        learning_rate: LR string (for plot filenames)
    """

    optimizer = AdamW([
    {'params': model.base_model.encoder.layer[-2:].parameters(), 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': learning_rate}
])

    # 1. Training
    trained_model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        patience=15,
        epochs=100,
        num_classes=num_classes
    )
    
    # 2. Plot training curves
    plot_training_history(history, method=method_name, lr=learning_rate)
    
    # 3. Final evaluation on test set
    test_metrics = evaluate_model(
        model=trained_model,
        data_loader=test_loader,
        criterion=nn.CrossEntropyLoss(),
        device=device,
        num_classes=num_classes
    )
    
    # 4. Plot confusion matrix
    class_names = ["Low", "Lower-Middle", "Upper-Middle", "High"]
    plot_confusion_matrix(
        test_metrics['confusion_matrix'], 
        classes=class_names,
        method=method_name,
        lr=learning_rate
    )
    
    # 5. Print final results
    print("\n" + "="*50)
    print("FINAL TEST RESULTS")
    print(f"Test Loss: {test_metrics['loss']:.4f}")
    print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Test F1: {test_metrics['f1_score']:.4f}")
    print("\nClassification Report:")
    print(test_metrics['report'])
    
    return {
        'model': trained_model,
        'history': history,
        'test_metrics': test_metrics
    }

# 1. Load

In [11]:
def set_seed(seed=42):
    import random, torch, numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [12]:
base_path_alvaro = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project")

base_path = base_path_alvaro # change according to user

In [11]:
# Step 1. Call the data
final_df = pd.read_csv(base_path / "Final_df_extended.csv")

# Step 2. Select only the climate related speeches
df_sentiment = final_df[final_df['contains_climate_keyword'] != False]

# Step 3. Keep only the most important columns
df_climate_change = df_sentiment.drop(['Session', 'Speech', 'number_sentences', 'number_tokens', 'cleaned_speeches_postagging_expanded', 'contains_climate_keyword', 'climate_sentences', 'contains_climate_keyword', 'matched_climate_keywords', 'climate_sentences_extended'], axis = 1)


# Run Once - Already Run

In [16]:
tqdm.pandas()
# Step 1: Replace bigrams
df_climate_change["speeches_for_keyword_search"] = df_climate_change["speeches_for_keyword_search"].progress_apply(
    lambda x: replace_bigrams(x, bigrams)
)

# Step 2. Select only those speeches that mention CLIMATE_CHANGE
df_climate_change[['matched_climate_keywords', 'contains_climate_keyword']] = df_climate_change['speeches_for_keyword_search'].progress_apply(
    lambda text: extract_keyword_info(text, 'climate_change')
)

# Step 3. Only select those instances with climate = True
df_climate_change_smp = df_climate_change[df_climate_change['contains_climate_keyword'] == True]

# Step 4. Extend the climate sentences WATCH OUT WITH THE HEAD HERE LIL BOI
df_climate_change_smp['climate_sentences_extended'] = df_climate_change_smp.progress_apply(
    lambda row: extract_climate_sentences_with_context(row['speeches_for_keyword_search'], row['matched_climate_keywords']),
    axis=1
)

# Step 7: Turn climate_sentences_extended into a STR and not a list
df_climate_change_smp['climate_sentences_extended'] = df_climate_change_smp['climate_sentences_extended'].apply(
    lambda sentences: ' '.join(sentences) if isinstance(sentences, list) else sentences
)

df_climate_change_smp["count_climate_change"] = df_climate_change_smp["speeches_for_keyword_search"].str.count("climate_change")

df_climate_change_smp

100%|██████████| 2946/2946 [16:21<00:00,  3.00it/s]
C:\Users\alvar\AppData\Local\Temp\ipykernel_440\1043948399.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_climate_change_smp['climate_sentences_extended'] = df_climate_change_smp.progress_apply(
C:\Users\alvar\AppData\Local\Temp\ipykernel_440\1043948399.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_climate_change_smp['climate_sentences_extended'] = df_climate_change_smp['climate_sentences_extended'].apply(
C:\Users\alvar\AppData\Local\Te

,Year,ISO-Code,Income Level,speeches_for_keyword_search,matched_climate_keywords,contains_climate_keyword,climate_sentences_extended,count_climate_change
4,1990,BGD,1,mr. president warm felicitations are due you o...,[climate_change],True,the conference must produce results that will ...,1
6,1990,CHN,1,i should like to begin by warmly congratulatin...,[climate_change],True,looking forward into the 1990s we see a world ...,1
8,1990,COD,1,mr. president the forty-fifth session of the u...,[climate_change],True,an increase in the planet's average temperatur...,1
14,1990,DNK,4,i congratulate you sir on your election as pre...,[climate_change],True,at the same time we must not lose sight of oth...,4
21,1990,ISL,4,mr. president to congratulate you on your elec...,[climate_change],True,international treaties in specific fields of t...,1
...,...,...,...,...,...,...,...,...
3375,2024,WSM,2,excellencies i extend my congratulations to hi...,[climate_change],True,please be assured of samoas support in the suc...,7
3376,2024,YEM,1,ladies and gentlemen it is a happy coincidence...,[climate_change],True,for this reason the republic of yemen renews i...,1
3377,2024,ZAF,3,president of the 79th session of the un genera...,[climate_change],True,extreme_weather such as flooding fires and dro...,2
3378,2024,ZMB,2,ladies and gentlemen i congratulate you your e...,[climate_change],True,furthermore zambia recognises the efforts of h...,2


In [18]:
df = get_text(df_climate_change_smp.copy(), entry_1='speeches_for_keyword_search', entry_2='climate_sentences_extended',entry_3='count_climate_change')
df

100%|██████████| 2946/2946 [00:00<00:00, 63489.33it/s]


,Year,ISO-Code,Income Level,speeches_for_keyword_search,matched_climate_keywords,contains_climate_keyword,climate_sentences_extended,count_climate_change,final_text
4,1990,BGD,1,mr. president warm felicitations are due you o...,[climate_change],True,the conference must produce results that will ...,1,the conference must produce results that will ...
6,1990,CHN,1,i should like to begin by warmly congratulatin...,[climate_change],True,looking forward into the 1990s we see a world ...,1,looking forward into the 1990s we see a world ...
8,1990,COD,1,mr. president the forty-fifth session of the u...,[climate_change],True,an increase in the planet's average temperatur...,1,an increase in the planet's average temperatur...
14,1990,DNK,4,i congratulate you sir on your election as pre...,[climate_change],True,at the same time we must not lose sight of oth...,4,i congratulate you sir on your election as pre...
21,1990,ISL,4,mr. president to congratulate you on your elec...,[climate_change],True,international treaties in specific fields of t...,1,international treaties in specific fields of t...
...,...,...,...,...,...,...,...,...,...
3375,2024,WSM,2,excellencies i extend my congratulations to hi...,[climate_change],True,please be assured of samoas support in the suc...,7,excellencies i extend my congratulations to hi...
3376,2024,YEM,1,ladies and gentlemen it is a happy coincidence...,[climate_change],True,for this reason the republic of yemen renews i...,1,for this reason the republic of yemen renews i...
3377,2024,ZAF,3,president of the 79th session of the un genera...,[climate_change],True,extreme_weather such as flooding fires and dro...,2,president of the 79th session of the un genera...
3378,2024,ZMB,2,ladies and gentlemen i congratulate you your e...,[climate_change],True,furthermore zambia recognises the efforts of h...,2,ladies and gentlemen i congratulate you your e...


# Start Here

In [69]:
# Load ClimateBERT
model_name = "Climatebert/distilroberta-base-climate-f"
base_model = AutoModel.from_pretrained(model_name)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4,output_attentions = False, output_hidden_states = False)


Some weights of RobertaModel were not initialized from the model checkpoint at Climatebert/distilroberta-base-climate-f and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at Climatebert/distilroberta-base-climate-f and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [70]:
#df.to_csv(base_path_alvaro / "DF_bert_embeddings.csv", index=False, encoding='utf-8')
df = pd.read_csv(base_path_alvaro / "DF_bert_embeddings.csv")
df_1 = df.drop(['climate_sentences_extended', 'contains_climate_keyword', 'matched_climate_keywords', 'speeches_for_keyword_search', 'Year', 'ISO-Code', 'count_climate_change'], axis = 1)

df_1['Income Level Encoded'] = df_1['Income Level'] - 1

df_1 = df_1.drop(['Income Level'], axis = 1)

df_1

,final_text,Income Level Encoded
0,the conference must produce results that will ...,0
1,looking forward into the 1990s we see a world ...,0
2,an increase in the planet's average temperatur...,0
3,i congratulate you sir on your election as pre...,3
4,international treaties in specific fields of t...,3
...,...,...
2941,excellencies i extend my congratulations to hi...,1
2942,for this reason the republic of yemen renews i...,0
2943,president of the 79th session of the un genera...,2
2944,ladies and gentlemen i congratulate you your e...,1


In [71]:
class ClimateDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# First split: 80% train+val, 20% test (unchanged)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df_1['final_text'].tolist(), 
    df_1['Income Level Encoded'].tolist(), 
    test_size=0.2, 
    random_state=42,
    stratify=df_1['Income Level Encoded']
)

# Second split: 80% train, 20% validation (of the remaining 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,  # 0.25 * 0.8 = 0.2 of original
    random_state=42,
    stratify=y_trainval
)

train_dataset = ClimateDataset(X_train, y_train, tokenizer)
val_dataset = ClimateDataset(X_val, y_val, tokenizer)
test_dataset = ClimateDataset(X_test, y_test, tokenizer)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [72]:
import torch
import torch.nn as nn

class ClimateBERTClassifier(nn.Module):
    def __init__(self, base_model, num_classes=4):
        super(ClimateBERTClassifier, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(base_model.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, 
                                attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Re-initialize base model and classifier
# Re-initialize base model and classifier
model = ClimateBERTClassifier(base_model).to(device)

# Correct reference to the model used
# Freeze all layers first
for param in model.base_model.parameters():
    param.requires_grad = False

# Unfreeze classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# After one training step, check gradients
for name, param in model.named_parameters():
    if name.startswith("base_model.encoder.layer.4.") or name.startswith("base_model.encoder.layer.5.") or name.startswith("classifier"):
        print(f"{'Trainable' if param.requires_grad else 'Frozen'}: {name}")


Frozen: base_model.encoder.layer.4.attention.self.query.weight
Frozen: base_model.encoder.layer.4.attention.self.query.bias
Frozen: base_model.encoder.layer.4.attention.self.key.weight
Frozen: base_model.encoder.layer.4.attention.self.key.bias
Frozen: base_model.encoder.layer.4.attention.self.value.weight
Frozen: base_model.encoder.layer.4.attention.self.value.bias
Frozen: base_model.encoder.layer.4.attention.output.dense.weight
Frozen: base_model.encoder.layer.4.attention.output.dense.bias
Frozen: base_model.encoder.layer.4.attention.output.LayerNorm.weight
Frozen: base_model.encoder.layer.4.attention.output.LayerNorm.bias
Frozen: base_model.encoder.layer.4.intermediate.dense.weight
Frozen: base_model.encoder.layer.4.intermediate.dense.bias
Frozen: base_model.encoder.layer.4.output.dense.weight
Frozen: base_model.encoder.layer.4.output.dense.bias
Frozen: base_model.encoder.layer.4.output.LayerNorm.weight
Frozen: base_model.encoder.layer.4.output.LayerNorm.bias
Frozen: base_model.encod

In [74]:
learning_rates = [1e-3, 3e-4, 2e-4, 1e-5]
num_epochs = 100

all_histories = {}

for learning_rate in learning_rates:
    print(f"\n========== Starting run with LR: {learning_rate} ==========\n")
    
    # Reset seed and clear cache
    set_seed(42)
    torch.cuda.empty_cache()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Reinitialize model
    model = ClimateBERTClassifier(base_model).to(device)

    # Freeze all layers
    for param in model.base_model.parameters():
        param.requires_grad = False

    # Unfreeze classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    optimizer = AdamW([
        {'params': model.base_model.encoder.layer[-1:].parameters(), 'lr': 2e-5},
        {'params': model.classifier.parameters(), 'lr': learning_rate}
    ])

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=len(train_loader) * num_epochs
    )

    trained_model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        patience=15,
        epochs=num_epochs,
        num_classes=4
    )

    all_histories[learning_rate] = history

    plot_training_history(history, method="ClimateBERT", lr=learning_rate)


    test_metrics = evaluate_model(
        model=trained_model,
        data_loader=test_loader,
        criterion=nn.CrossEntropyLoss(),
        device=device,
        num_classes=4
    )

    class_names = ["Low", "Lower-Middle", "Upper-Middle", "High"]
    plot_confusion_matrix(test_metrics['conf_matrix'], classes=class_names, method="ClimateBERT", lr=learning_rate)

    # 5. Print final results
    print("\n" + "="*50)
    print("FINAL TEST RESULTS")
    print(f"Test Loss: {test_metrics['loss']:.4f}")
    print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Test F1: {test_metrics['f1_score']:.4f}")
    print("\nClassification Report:")
    print(test_metrics['report'])



========== Starting run with LR: 0.001 ==========

This is the epoch number 0
Epoch 1:
Train  - Loss: 1.3589, Acc: 0.3107, F1: 0.2418
Val    - Loss: 1.3334, Acc: 0.3260, F1: 0.2148
--------------------------------------------------
This is the epoch number 1


KeyboardInterrupt: 

In [ ]:
""" set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rates = [1e-3, 3e-4, 2e-4, 1e-5]
# Initialize model

for lr in learning_rates:
    print(f"\n===== Starting run with classifier LR: {lr} =====")

    # Re-initialize model each time
    model = ClimateBERTClassifier(base_model).to(device)

    optimizer_1 = AdamW([
        {'params': model.base_model.encoder.layer[-2:].parameters(), 'lr': 2e-5},
        {'params': model.classifier.parameters(), 'lr': lr}
    ])

    scheduler_1 = get_linear_schedule_with_warmup(
        optimizer_1,
        num_warmup_steps=0,
        num_training_steps=len(train_loader)*100
    )

    run_full_experiment(model, train_loader, val_loader, test_loader,
                        device,
                        optimizer=optimizer_1,
                        scheduler=scheduler_1,
                        learning_rate=lr)
 """


===== Starting run with classifier LR: 0.001 =====


KeyboardInterrupt: 

In [ ]:
""" # Initialize model and optimizer
model = ClimateBERTClassifier(base_model).to(device)
optimizer = AdamW([
    {'params': model.base_model.encoder.layer[-2:].parameters(), 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
])
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader)*100
)

# Train the model
best_model_path, training_stats = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    patience=15,
    epochs=100
)

# Evaluate on test set
model.load_state_dict(torch.load(best_model_path))
test_loss, test_acc, test_preds, test_labels = evaluate_model(
    model=model,
    data_loader=test_loader,
    device=device,
    return_predictions=True
)

print(f"Test Accuracy: {test_acc:.4f}")
print(classification_report(test_labels, test_preds))  """